### Setup

The experiment name groups every run of this project in one place. MLflow creates it on first use.

In [1]:
import os
import mlflow
import mlflow.sklearn
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score,
                             ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE

mlflow.set_experiment("classification_clients")


2026/10/09 14:47:47 INFO mlflow.tracking.fluent: Experiment with name 'classification_clients' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/kcrowly/PycharmProjects/CardProfiler/notebooks/Classification/mlruns/2', creation_time=1791557267411, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1791557267411, lifecycle_stage='active', name='classification_clients', tags={}, trace_location=None, workspace='default'>

### Load the cleaned data

The 7 raw behavioural variables predict `target`. No clustering column, no log1p — that preprocessing belongs to the clustering side.

In [2]:
df = pd.read_csv("../../data/processed/df_clean_clustered.csv")
feats = ["BALANCE", "PURCHASES", "ONEOFF_PURCHASES",
         "INSTALLMENTS_PURCHASES", "CASH_ADVANCE", "CREDIT_LIMIT", "PAYMENTS"]
X = df[feats]
y = df["target"]


### Split and balance

Stratified split, then SMOTE on the training set only — the test set keeps the real class proportions, otherwise the scores flatter the models.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)

X_train, y_train = SMOTE(random_state=0).fit_resample(X_train, y_train)

print("train:", X_train.shape, "| test:", X_test.shape)


train: (8142, 7) | test: (1790, 7)


### The four pipelines

Same as FS5, with the GridSearch winners: only the `classifier` step changes, the scaler is identical everywhere.

In [4]:
pipelines = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(C=10, max_iter=1000, random_state=0)),
    ]),
    "DecisionTree": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", DecisionTreeClassifier(random_state=0)),
    ]),
    "RandomForest": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", RandomForestClassifier(n_estimators=200, max_depth=20, random_state=0)),
    ]),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(C=10, random_state=0)),
    ]),
}


### One run per model

The `with mlflow.start_run()` block is the run: everything logged inside belongs to that training. Params explain the differences between runs, the confusion matrix keeps a visual trace, and the pipeline is stored whole so it can be reloaded later.

The loop retrains on purpose — that is what a run records.

In [5]:
os.makedirs("../../reports/mlflow", exist_ok=True)

for name, pipeline in pipelines.items():
    with mlflow.start_run(run_name=name):
        pipeline.fit(X_train, y_train)
        pred = pipeline.predict(X_test)

        mlflow.log_param("model", name)
        clf_params = pipeline.named_steps["classifier"].get_params()
        for key in ["n_estimators", "max_depth", "C"]:
            if key in clf_params:
                mlflow.log_param(key, str(clf_params[key]))

        mlflow.log_metric("accuracy", accuracy_score(y_test, pred))
        mlflow.log_metric("precision", precision_score(y_test, pred, average="weighted"))
        mlflow.log_metric("recall", recall_score(y_test, pred, average="weighted"))
        mlflow.log_metric("f1", f1_score(y_test, pred, average="weighted"))

        fig, ax = plt.subplots()
        ConfusionMatrixDisplay.from_estimator(pipeline, X_test, y_test,
                                              ax=ax, cmap="Blues", colorbar=False)
        cm_path = f"../../reports/mlflow/CM_{name}.png"
        fig.savefig(cm_path, bbox_inches="tight")
        plt.close(fig)
        mlflow.log_artifact(cm_path)

        mlflow.sklearn.log_model(pipeline, name="pipeline",
                                 skops_trusted_types=["sklearn.tree._tree.Tree"])
        print(name, "F1:", round(f1_score(y_test, pred, average="weighted"), 4))


2026/10/09 14:47:51 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


LogisticRegression F1: 0.9066


2026/10/09 14:47:54 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


DecisionTree F1: 0.9425


2026/10/09 14:47:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


RandomForest F1: 0.9732


2026/10/09 14:48:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


SVM F1: 0.9396


### Compare the runs

The same table `mlflow ui` displays, sorted by F1 descending.

In [6]:
exp = mlflow.get_experiment_by_name("classification_clients")
runs = mlflow.search_runs(experiment_ids=exp.experiment_id)
runs = runs.sort_values("metrics.f1", ascending=False)
runs[["tags.mlflow.runName", "metrics.accuracy", "metrics.precision",
      "metrics.recall", "metrics.f1"]]


,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1
1,RandomForest,0.973184,0.973297,0.973184,0.973182
2,DecisionTree,0.942458,0.942549,0.942458,0.942497
0,SVM,0.939665,0.939948,0.939665,0.939626
3,LogisticRegression,0.906704,0.906935,0.906704,0.906570
